## 과제: 로컬 모델을 활용한 RAG 시스템 구축

**개요**

지난 과제는 기본적인 RAG 프로세스를 구축하였습니다. 가장 기본적인 모듈과 쉽게 구현할 수 있는 방식으로요!

하지만, 우리는 때로는 외부 모델을 사용하거나, Local LLM을 사용하기도 합니다.

혹은, 더 나은 방식의 embedding 모델을 사용하기도 합니다.

이번 과제는 지난 번 주어진 BASIC RAG 파이프라인에서 아래의 요구사항에 따라 주요 모듈을 변경하는 것이 목표입니다!

## Task 1: OpenAI LLM 과 Embedding 모델을 탈피하기!

여러분 OpenAI 의 임베딩 모델과 LLM 은 사용할 때마다 과금이 되고 있습니다.

이번에 여러분께 주어진 임무는 과금 0원의 RAG 를 만들어 보는 것입니다.

그러기 위해서는 아래의 요구사항을 충족하는 코드를 작성해야 합니다.

1. 임베딩 모델을 변경하기(임베딩 강의를 참고하여 본인의 상황에 맞는 무료 모델을 선택하세요)
2. LLM 모델을 변경하기(모델 강의를 참고하여 본인의 상황에 맞는 무료 모델을 선택하세요)

**실습데이터**

소프트웨어정책연구소(SPRi) - 2023년 12월호

- 저자: 유재흥(AI정책연구실 책임연구원), 이지수(AI정책연구실 위촉연구원)
- 링크: https://spri.kr/posts/view/23669
- 파일명: `SPRI_AI_Brief_2023년12월호_F.pdf`

**안내**

> 아래는 기본 코드 입니다. 아래 코드에서 여러분들이 변경하시면 됩니다.

In [68]:
!pip install langchain-text-splitters
!pip install langchain-community pdfplumber pymupdf

In [69]:
!pip install faiss-gpu
!pip install faiss-cpu
!pip install langchain-openai sentence-transformers

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [70]:
import os
os.environ["OPENAI_API_KEY"] = "sk-proj-..."
print(os.environ.get("OPENAI_API_KEY"))

sk-proj-...

In [86]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader, PDFPlumberLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
## from langchain_openai import ChatOpenAI, OpenAIEmbeddings # OpenAI 모델은 과금 문제로 주석처리.
## 무료 임베딩 모델로 변경.
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# 단계 1: 문서 로드(Load Documents)
loader = PDFPlumberLoader("/content/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

# 단계 3: 임베딩(Embedding) 생성
# OpenAIEmbeddings 대신 HuggingFaceEmbeddings를 사용.
## 원하는 모델을 지정. 여기서는 'sentence-transformers/multi-qa-distilbert-cos-v1'를 예시로 사용.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/multi-qa-distilbert-cos-v1")

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
# LLM의 Context Window를 고려하여 검색할 문서 수를 감소.
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following context to answer the question accurately.
If the answer is not in the context, say '모르겠습니다.'
Answer in Korean.

#Question:
{question}

#Context:
{context}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
# LLM도 무료 모델로 변경. (예: Ollama, HuggingFacePipeline 등)
# 현재는 ChatOpenAI를 그대로 두지만, 과제 1의 요구사항에 따라 변경.
## 여기에 무료 LLM(HuggingFacePipeline을 설정.

pipe = pipeline(
    "text-generation",
    model="LiquidAI/LFM2.5-1.2B-Instruct",
    trust_remote_code=True,
    max_length=512,
    temperature=0.1,
    do_sample=True,
)

llm = HuggingFacePipeline(pipeline=pipe)

# 단계 8: 체인(Chain) 생성
# LLM을 설정한 후 이 부분을 실행할 수 있습니다.
chain = (
     {"context": retriever, "question": RunnablePassthrough()}
     | prompt
     | llm
     | StrOutputParser()
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Device set to use cpu


## Task 2: Text Splitter 선택

지금까지 `RecursiveCharacterTextSplitter` 만을 사용하여 텍스트를 분할하였습니다.

이제는 여러분이 선택한 Text Splitter 를 사용하여 문서를 분할해 보세요.

**참고**

`SemanticChunker` 를 꼭 적용해 보세요!

In [72]:
## 사전 코드
!pip install langchain-experimental

In [87]:
# Your Code Here
from langchain_experimental.text_splitter import SemanticChunker

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
)

split_documents = semantic_splitter.split_documents(docs)

vectorstore = FAISS.from_documents(
    documents=split_documents,
    embedding=embeddings
)

In [88]:
## Test Code1 : Sematic Chunker 자체 검증
print(f"총 문서 개수: {len(docs)}")
print(f"SemanticChunker 분할 후 chunk 수: {len(split_documents)}")

# 앞부분 3개 chunk 내용 확인.
for i in range(3):
    print(f"\n--- Chunk {i+1} ---")
    print(split_documents[i].page_content[:500])

총 문서 개수: 23
SemanticChunker 분할 후 chunk 수: 35

--- Chunk 1 ---
12
2023년 월호


--- Chunk 2 ---
2023년 12월호
Ⅰ
. 인공지능 산업 동향 브리프
1. 정책/법제
▹ 미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표 ·························1
▹ G7, 히로시마 AI 프로세스를 통해 AI 기업 대상 국제 행동강령에 합의···························2
▹ 영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언···························3
▹ 미국 법원, 예술가들이 생성 AI 기업에 제기한 저작권 소송 기각·····································4
▹ 미국 연방거래위원회, 저작권청에 소비자 보호와 경쟁 측면의 AI 의견서 제출·················5
▹ EU AI 법 3자 협상, 기반모델 규제 관련 견해차로 난항···················································6
2. 기업/산업
▹ 미국 프런티어 모

--- Chunk 3 ---
주요 행사
▹CES 2024·····························································································································19
▹AIMLA 2024·························································································································19
▹AAAI Conference on Artificial Intelligence··································································19


In [89]:
# Test Code2 : Retriever 정상 동작 테스트 (RAG 전 단계)

query = "AI 정책과 관련된 주요 이슈는 무엇인가?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- 검색 결과 {i+1} ---")
    print(doc.page_content[:500])


--- 검색 결과 1 ---
홈페이지 : https://spri.kr/
보고서와 관련된 문의는 AI정책연구실(jayoo@spri.kr, 031-739-7352)으로 연락주시기 바랍니다.


In [91]:
# Test Code3 : RAG End-to-End 테스트
question_not_in_doc = "회의에 참가한 주요국가의 갯수는 몇 개지?"
response_not_in_doc = chain.invoke(question_not_in_doc)
print(response_not_in_doc)

You are an assistant for question-answering tasks.
Use the following context to answer the question accurately.
If the answer is not in the context, say '모르겠습니다.'
Answer in Korean.

#Question:
회의에 참가한 주요국가의 갯수는 몇 개지?

#Context:
[Document(id='5b69d5a6-1cce-4eac-8d79-ee1ae1e44a10', metadata={'source': '/content/SPRI_AI_Brief_2023년12월호_F.pdf', 'file_path': '/content/SPRI_AI_Brief_2023년12월호_F.pdf', 'page': 22, 'total_pages': 23, 'Author': 'dj', 'Creator': 'Hwp 2018 10.0.0.13462', 'Producer': 'Hancom PDF 1.3.0.542', 'CreationDate': "D:20231208132838+09'00'", 'ModDate': "D:20231208132838+09'00'", 'PDFVersion': '1.4'}, page_content='홈페이지 : https://spri.kr/\n보고서와 관련된 문의는 AI정책연구실(jayoo@spri.kr, 031-739-7352)으로 연락주시기 바랍니다.')]

#Answer: 5개의 주요국가가 회의의 참가자였습니다.


## Task 3. 스트림릿에 100% 무료로 동작하는 RAG

여러분이 만든 100% 무료 RAG 를 스트림릿에 배포해 보세요!

코드는 별고의 `.py` 파일에 작성해 주세요!

**주요 기능**

- PDF 파일 업로드
- FAISS 벡터스토어에 데이터 저장
- 업로드한 파일 기반 Q&A 챗봇
- 유료 모델 사용금지!!